# Customer Churn & Retention Analysis
## Phase 2: Python Data Cleaning & Exploratory Data Analysis (EDA)

**Author:** Data Analyst Portfolio Project  
**Target Roles:** Entry-Level Data Analyst / Business Analyst / Reporting Analyst  
**Tools:** Python (`pandas`, `numpy`, `matplotlib`, `seaborn`)  

---
### Project Objective
Analyze customer demographics, account characteristics, service subscriptions, and billing methods to uncover churn drivers, calculate key retention metrics, and isolate high-risk customer segments.

### Step 1: Import Libraries and Configure Environment

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean visualization aesthetics
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)

### Step 2: Load Raw Dataset
We load the immutable raw dataset directly from `../data/raw/Telco-Customer-Churn.csv`.

In [ ]:
raw_path = os.path.join('..', 'data', 'raw', 'Telco-Customer-Churn.csv')
df = pd.read_csv(raw_path)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

### Step 3: Inspect Missing Values & Clean `TotalCharges`
In Phase 1, we discovered that `TotalCharges` has 11 records containing a whitespace string (`' '`) instead of numeric digits. All 11 records correspond to new customers with `tenure = 0` months. We convert them to `0.0` and cast to float.

In [ ]:
# Detect whitespace in TotalCharges
whitespace_mask = df['TotalCharges'].str.strip() == ''
print(f"Whitespace rows in TotalCharges: {whitespace_mask.sum()}")

# Clean TotalCharges: replace blanks with '0.0' and cast to float64
df.loc[whitespace_mask, 'TotalCharges'] = '0.0'
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
print(f"TotalCharges data type is now: {df['TotalCharges'].dtype}")

### Step 4: Feature Engineering
We create intuitive analytical features:
1. `ChurnNumeric`: Binary 1 / 0 for statistical aggregation.
2. `SeniorCitizen_Label`: Descriptive 'Yes'/'No' labels.
3. `TenureCohort`: Customer lifecycle brackets (0-12m, 13-24m, 25-48m, 49-60m, 61-72m).
4. `MonthlyChargeTier`: Pricing brackets (Low, Medium, High, Very High).

In [ ]:
df['ChurnNumeric'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)
df['SeniorCitizen_Label'] = df['SeniorCitizen'].apply(lambda x: 'Yes' if x == 1 else 'No')

tenure_bins = [-1, 12, 24, 48, 60, 72]
tenure_labels = ['0-12 Mos', '13-24 Mos', '25-48 Mos', '49-60 Mos', '61-72 Mos']
df['TenureCohort'] = pd.cut(df['tenure'], bins=tenure_bins, labels=tenure_labels)

charge_bins = [0, 35, 70, 90, 150]
charge_labels = ['Low (<$35)', 'Medium ($35-$70)', 'High ($70-$90)', 'Very High (>$90)']
df['MonthlyChargeTier'] = pd.cut(df['MonthlyCharges'], bins=charge_bins, labels=charge_labels)

df[['customerID', 'tenure', 'TenureCohort', 'MonthlyCharges', 'MonthlyChargeTier', 'TotalCharges', 'ChurnNumeric']].head()

### Step 5: Baseline Retention & Churn KPIs

In [ ]:
total_customers = len(df)
churned = df['ChurnNumeric'].sum()
retained = total_customers - churned
churn_rate = (churned / total_customers) * 100

print(f"Total Customers    : {total_customers:,}")
print(f"Retained Customers : {retained:,} ({retained/total_customers*100:.2f}%)")
print(f"Churned Customers  : {churned:,} ({churn_rate:.2f}%)")
print(f"Overall Churn Rate : {churn_rate:.2f}%")

### Step 6: Bivariate Exploratory Analysis
We define a reusable function to compute customer counts, churn counts, and churn rates across any segment.

In [ ]:
def analyze_segment(col):
    res = df.groupby(col, observed=False).agg(
        Total_Customers=('customerID', 'count'),
        Churned_Customers=('ChurnNumeric', 'sum')
    ).reset_index()
    res['Retained_Customers'] = res['Total_Customers'] - res['Churned_Customers']
    res['Churn_Rate_%'] = (res['Churned_Customers'] / res['Total_Customers'] * 100).round(2)
    res['Customer_Share_%'] = (res['Total_Customers'] / total_customers * 100).round(2)
    return res.sort_values('Churn_Rate_%', ascending=False)

# Display Contract analysis
analyze_segment('Contract')

### Step 7: Visualizing Churn Drivers
Below we generate the key diagnostic plots.

In [ ]:
# Plot Churn by Contract Type
contract_df = analyze_segment('Contract')
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=contract_df, x='Contract', y='Churn_Rate_%', palette='Reds_r', ax=ax)
ax.axhline(churn_rate, color='black', linestyle='--', label=f'Avg Churn Rate ({churn_rate:.1f}%)')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, weight='bold', xytext=(0, 4), textcoords='offset points')
ax.set_title('Churn Rate by Contract Type', fontsize=14, weight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 50)
ax.legend()
plt.show()

In [ ]:
# Plot Churn by Tenure Cohort
tenure_df = analyze_segment('TenureCohort')
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.barplot(data=tenure_df, x='TenureCohort', y='Churn_Rate_%', palette='Blues_r', ax=ax)
ax.axhline(churn_rate, color='red', linestyle='--', label=f'Avg Churn Rate ({churn_rate:.1f}%)')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, weight='bold', xytext=(0, 4), textcoords='offset points')
ax.set_title('Churn Rate by Customer Tenure Cohort', fontsize=14, weight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)
ax.legend()
plt.show()

### Step 8: Export Cleaned Dataset for MySQL & Power BI

In [ ]:
processed_path = os.path.join('..', 'data', 'processed', 'telco_churn_cleaned.csv')
df.to_csv(processed_path, index=False)
print(f"Cleaned dataset exported to: {processed_path}")
print(f"Final dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns, 0 nulls.")